## Preparation

In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [3]:
len(files), files[0]

(72,
 RawRepositoryFile(filename='01-agentic-rag/lessons/01-intro.md', content='# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "Ho

In [4]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [6]:
len(documents), documents[0]

(72,
 {'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses 

## Q1. How many lesson pages

How many lesson pages are in the dataset?

* 24
* 72
* 240
* 720

In [7]:
# Answer: 72

## Q2. Indexing and searching

Index the documents with minsearch - make `content` a text field and
`filename` a keyword field. Then search with this query:

> How does the agentic loop keep calling the model until it stops?

What's the `filename` of the first result?

* `01-agentic-rag/lessons/03-rag.md`
* `01-agentic-rag/lessons/14-agentic-loop.md`
* `04-evaluation/lessons/13-llm-as-judge.md`
* `06-best-practices/lessons/02-hybrid-search.md`

In [8]:
from minsearch import Index

index = Index(
    text_fields=['content'],
    keyword_fields=['filename']
)
index.fit(documents)

In [9]:
question = "How does the agentic loop keep calling the model until it stops?"

search_results = index.search(
    question,
    num_results=5
)

search_results

[{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry

In [10]:
# Answer: 01-agentic-rag/lessons/14-agentic-loop.md

## Q3. RAG

Now we will build a RAG assistant on top of this data. Let's use the rag helper 
script we prepared during the lessons:

```bash
wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
```

`RAGBase` was written for the FAQ schema (`section`/`question`/`answer`),
while our documents have `filename` and `content`.

Two solutions are possible:

- Implement the RAG flow yourself
- Take `RAGBase` and change the parts related to the FAQ schema - `search` (to use our index) and `build_context`

Build a RAG over the index from Q2 and answer the query:

> How does the agentic loop keep calling the model until it stops?

Use gpt-5.4-mini. How many input (prompt) tokens did we send to the model for
this request?

* 700
* 7000
* 70000
* 700000

We count input tokens instead of price because the cost depends on the model
and provider you use, but the size of the prompt we send is the same for
everyone.

Most LLM APIs report token usage on the response object (e.g.
`response.usage.input_tokens` / `prompt_tokens`). We'll read the input tokens
from there.

You will need to modify the code for the rag helper to expose the usage.

In the RAG Helper class, `llm` returns only the text. Modify it to return the whole response, and change `rag` to return both the answer and usage (as a tuple or create a small dataclass for that).

Note: for this question and the next ones, if your answer doesn't match exactly,
just select the closest option - especially if you use a different model or a
different LLM provider.

In [11]:
!curl -O https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py

  % Total    % Received % Xferd  Average Speed  Time    Time    Time   Current
                                 Dload  Upload  Total   Spent   Left   Speed

  0      0   0      0   0      0      0      0                              0
100   2134 100   2134   0      0   5615      0                              0
100   2134 100   2134   0      0   5614      0                              0
100   2134 100   2134   0      0   5612      0                              0


In [13]:
from rag_helper import RAGBase
from openai import OpenAI
import os
from dotenv import load_dotenv
load_dotenv()


openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

assistant = RAGBase(index, openai_client, model='qwen/qwen3-32b')


question = "How does the agentic loop keep calling the model until it stops?"
assistant.rag(question)

APIStatusError: Error code: 413 - {'error': {'message': 'Request too large for model `qwen/qwen3-32b` in organization `org_01kvpzwa2xedhvj2xp6kp7wd0a` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Requested 7196, please reduce your message size and try again. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [14]:
# Anthough I could not run the rag above, error message mentions tokens passed were 7196, so i'm sure of the approximate answer
# Answer: 7000

## Q4. Chunking

The lesson pages are long - some are thousands of characters. Long documents
make retrieval less precise: a match deep inside a page still pulls in the
whole page. A common fix is chunking: split each page into smaller,
overlapping pieces and index those instead.

gitsource has a helper for this: `chunk_documents`. It uses a sliding
window - a window of `size` characters slides across the text in steps of
`step` characters, and each window position becomes one chunk:

```python
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
```

With `size=2000` and `step=1000` (you can see the implementation
[here](https://github.com/alexeygrigorev/gitsource/blob/master/gitsource/chunking.py)):

- Each chunk is a window of `size` characters of the page.
- The window moves forward by `step` characters between chunks. Since `step`
  is smaller than `size`, consecutive chunks overlap by `size - step` (1000)
  characters, so a passage split across a boundary still appears whole in one
  of the chunks.
- Every chunk keeps the original fields (`filename`) and adds `start` (the
  offset in the page) and `content` (the chunk text).

How many chunks do you get?

* 70
* 295
* 1100
* 4500

In [15]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [17]:
len(chunks)

295

In [18]:
# Answer: 295

## Q5. RAG with chunking

Chunking makes each request smaller, because we send a smaller context to the
LLM. Let's measure that.

Index the chunks from Q4 (same as before: `content` as a text field,
`filename` as a keyword field), point your RAG at the chunk index, and
answer the same query again - reading the input tokens the same way as in Q3.

Compare the input tokens with Q3. How many fewer input tokens does the chunked
version send?

* about the same
* 3× fewer
* 10× fewer
* 30× fewer

In [20]:
index_v2 = Index(
    text_fields=['content'],
    keyword_fields=['filename']
)
index_v2.fit(chunks)

In [22]:
question = "How does the agentic loop keep calling the model until it stops?"

assistant = RAGBase(index_v2, openai_client, model='qwen/qwen3-32b')

question = "How does the agentic loop keep calling the model until it stops?"
assistant.rag(question)

('The agentic loop keeps calling the model until the model returns a response **without any function calls**, as explained in the code and context. Here\'s how it works:\n\n---\n\n### Core Loop Mechanism\n1. **Infinite Loop**: The loop uses a `while True` construct, continuously iterating until manually broken.\n2. **`has_function_calls` Flag**:  \n   At the start of each iteration, the `has_function_calls` flag is set to `False`. If the model\'s response contains **any function calls**, this flag is set to `True` and the tools are executed. If **no function calls** are detected after processing the response, the flag remains `False`.\n\n3. **Exit Condition**:  \n   If `has_function_calls == False` after processing a response, the loop breaks (`break`). This means the model has decided no further actions are needed and has **provided a final answer**.\n\n---\n\n### Key Code Flow\n```python\nwhile True:\n    # Assume has_function_calls = False at the start\n    # Model call and response

In [23]:
# total input tokens sent to llm is 2200 as compared to 7100 from before, so approximate answer is clear
# Answer: 3x fewer

## Q6. Turning it into an agent

So far search runs once, with the exact query. Let's make it agentic: give
the LLM a `search` tool and let it decide when (and what) to search. We
suggest [toyaikit](https://github.com/alexeygrigorev/toyaikit), the small
agent library from the module, but you can use anything you like - the OpenAI
Agents SDK, PydanticAI, LangChain, or a hand-written loop.

If you go with toyaikit:

```bash
uv add toyaikit
```

Create a `search` function that uses the chunk index. Give it a type hint and
a docstring - most frameworks read them to build the tool schema for you.

Build an agent with your `search` tool and run it (with toyaikit, the same way
as in the ToyAIKit lesson). Use these instructions for the agent (they nudge
it to search a few times):

> You're a course teaching assistant. Answer the student's question using the
> search tool. Make multiple searches with different keywords before answering.

Ask it:

> How does the agentic loop work, and how is it different from plain RAG?

The agent decides on its own when to search and when to answer. Count how many
times it called the `search` tool.

How many times did the agent call `search`?

> Note: the agent decides this itself, so it varies a little between runs -
> pick the closest option. We measured this with OpenAI `gpt-5.4-mini`; with a
> different model or provider the number may differ, so keep that in mind.

* 0
* 4
* 10
* 20

In [24]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [25]:
def search(query: str) -> dict[str, str]:
    """
    Search the course lesson database for entries matching the given query.
    """
    return index_v2.search(
        query,
        num_results=5
    )

agent_tools = Tools()
agent_tools.add_tool(search)

agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the course lesson database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [26]:
instructions = """
You're a course teaching assistant. Answer the student's question using the search tool.
Make multiple searches with different keywords before answering.
"""

question = "How does the agentic loop work, and how is it different from plain RAG?"

In [27]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model='qwen/qwen3-32b',
                            client=openai_client)
)

In [28]:
result = runner.loop(
    prompt=question,
    callback=callback,
)

-> Response received


-> Response received


Feature,Agentic Loop,Plain RAG
Workflow,"Iterative, decision-based","Linear, fixed pipeline"
Conversation History,Maintained automatically,Not retained
Tool Usage,Can call multiple tools sequentially,Typically uses search/retrieval only
Cost,Higher (multiple LLM/tool calls),Lower (single LLM call)
Use Cases,"Complex, multi-step tasks","Simple Q&A, document summarization"


c:\Valinor\Programs\miniconda3\envs\llm_zoomcamp\Lib\site-packages\toyaikit\chat\runners.py:283: UnknownModelWarning: No pricing data for model 'qwen/qwen3-32b'. Register it with PricingConfig.register_model(...) to get cost calculations.
  cost_info = self.pricing_config.calculate_cost(


In [30]:
result.all_messages

[EasyInputMessage(content="\nYou're a course teaching assistant. Answer the student's question using the search tool.\nMake multiple searches with different keywords before answering.\n", role='developer', phase=None, type=None),
 EasyInputMessage(content='How does the agentic loop work, and how is it different from plain RAG?', role='user', phase=None, type=None),
 ResponseReasoningItem(id='resp_01kvrrb99peyzax3jdzb6nn1vz', summary=[], type='reasoning', content=[Content(text="Okay, the user is asking about the agentic loop and how it's different from plain RAG. Let me start by recalling what I know about these concepts.\n\nRAG stands for Retrieval-Augmented Generation. It's a method where a model retrieves information from a database or documents and then uses that to generate a response. The main idea is to enhance the model's knowledge with up-to-date or specific information from external sources.\n\nNow, the agentic loop. Hmm, I remember that agents in AI systems are often designed

In [ ]:
# the search tool has been called once, so closest answer is 0
# Answer: 0

## End of Assignment